In [1]:
# Placeholder cell intentionally left minimal.
# Właściwa konfiguracja środowiska zaczyna się w kolejnej komórce kodu.

Niniejszy notatnik odpowiada za implementację matematycznego algorytmu bilansującego czynniki podróży według paradygmatu modelu podwójnie ograniczonego (Doubly-Constrained Spatial Interaction Model Wilsona).

> Zakres iteracji miast/warunków: `paris_core`, `dublin_core`, `warsaw_core`, `paris_extended`, `dublin_extended`, `warsaw_extended`.

> Obliczenia mogą być prowadzone dla wariantu `core`, `extended` albo równolegle dla obu wariantów (`both`), zgodnie z architekturą Etapu III.

Opiera się o koszty dojazdu wyliczone przez `r5py` w Etapie III ($c_{ij}$ interpretowane tu w ujęciu uogólnionym `cost_min_strict`), w zestawieniu wektorów rynkowych O/D. W celu sprowadzenia do homogenicznych wyników i uniknięcia nieporozumień z agregacjami różnych stref, silniki bilansujące wywoływane są *niezależnie dla poszczególnych aglomeracji i wariantów przestrzennych*.

---
**Uwagi badawcze i techniczne:**
* **NOWOŚĆ METODOLOGICZNA:** Zgodnie z wytycznymi wprowadzono trójwariantową iterację modeli grawitacyjnych. Modele sztywne A i B ($\beta=0.03$, $\beta=0.15$) zestawiane są z wariantem C (wrażliwość ramy do siatki zamożności miasta `wealth_index` / `income_median`).
* **Algorytm:** Obliczenia bazują na iteracyjnym algorytmie IPF (Iterative Proportional Fitting) bez wymogu scipy.optimize - wszystkie operacje wykonywane są na macierzach NumPy, co zwiększa czytelność i reproducibility.

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
from pathlib import Path
from datetime import datetime
import platform
from typing import Optional

# Ustalenie domyslnych sciezek (odporne na CWD kernela: repo root albo Netobooks/)
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs").exists() and (ROOT.parent / "outputs").exists():
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
VARIANT: str = 'both'  # 'core', 'extended', lub 'both'
VARIANTS: list = ['core', 'extended'] if VARIANT == 'both' else [VARIANT]

# INPUTS_E3 / OUTPUTS_E4 obliczane per-miasto wewnątrz run_city_gravity_pipeline
for _v in VARIANTS:
    (OUTPUTS / "etap4" / _v).mkdir(parents=True, exist_ok=True)

# Definicja miast na wywolanie iteracyjne (wszystkie warianty)
CITIES = [f"{base}_{v}" for v in VARIANTS for base in ['paris', 'dublin', 'warsaw']]

def _rel(p: Path) -> str:
    """Zwarca sciezke wzglednie do ROOT dla lzejszego indexu json."""
    try:
        return str(p.relative_to(ROOT))
    except ValueError:
        return str(p)


def _pick_first_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


print(f"Srodowisko gotowe: Python {platform.python_version()}")
print(f"ROOT: {ROOT}")

Srodowisko gotowe: Python 3.9.13
ROOT: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters


## Konstrukcja Silnika Modeu: Algorytm IPF (Doubly-Constrained)

In [3]:
def doubly_constrained_gravity(cost_matrix_df, O_col='O_pop', D_col='D_jobs', beta=0.08, tol=1e-5, max_iter=1500):
    """
    Implementuje proceduralny algorytm Wilsona oparty na Iterative Proportional Fitting (IPF).
    Obsługuje dynamiczne beta oraz naprawę wsparcia OD dla przypadków strukturalnie trudnych.
    """
    df = cost_matrix_df.copy()

    # 1. Funkcja impedancji
    if isinstance(beta, str) and beta in df.columns:
        df['f_cij'] = np.exp(-df[beta] * df['cost'])
    else:
        df['f_cij'] = np.exp(-beta * df['cost'])

    O_target = df.groupby('origin_id')[O_col].first().astype('float64')
    D_target = df.groupby('dest_id')[D_col].first().astype('float64')

    total_O = float(O_target.sum())
    total_D = float(D_target.sum())
    if total_D == 0 or total_O == 0:
        raise ValueError("Suma popytu lub podaży wynosi 0! Bilansowanie niemożliwe.")

    # 2. Odfiltrowanie wsparcia do dodatnich marginaliów
    positive_origins = set(O_target[O_target > 0].index)
    positive_dests = set(D_target[D_target > 0].index)
    df = df[df['origin_id'].isin(positive_origins) & df['dest_id'].isin(positive_dests)].copy()

    if df.empty:
        raise ValueError("Brak relacji OD po odfiltrowaniu dodatnich marginaliów.")

    # Czyszczenie izolacji w grafie dwudzielnym
    changed = True
    while changed:
        changed = False
        active_origins = set(df['origin_id'].unique())
        active_dests = set(df['dest_id'].unique())

        origin_deg = df.groupby('origin_id').size()
        dest_deg = df.groupby('dest_id').size()
        non_isolated_origins = set(origin_deg[origin_deg > 0].index)
        non_isolated_dests = set(dest_deg[dest_deg > 0].index)

        if non_isolated_origins != active_origins or non_isolated_dests != active_dests:
            df = df[df['origin_id'].isin(non_isolated_origins) & df['dest_id'].isin(non_isolated_dests)].copy()
            changed = True

    if df.empty:
        raise ValueError("Brak relacji OD po czyszczeniu wsparcia bipartytowego.")

    O_target = O_target.reindex(df['origin_id'].unique()).fillna(0)
    D_target = D_target.reindex(df['dest_id'].unique()).fillna(0)

    # 3. Bilansowanie podaży per spójny komponent grafu origin-dest
    comp_origin_to_dests = df.groupby('origin_id')['dest_id'].agg(set).to_dict()
    comp_dest_to_origins = df.groupby('dest_id')['origin_id'].agg(set).to_dict()

    unvisited_origins = set(comp_origin_to_dests.keys())
    components = []

    while unvisited_origins:
        seed_origin = unvisited_origins.pop()
        stack_origins = [seed_origin]
        seen_origins = set()
        seen_dests = set()

        while stack_origins:
            oid = stack_origins.pop()
            if oid in seen_origins:
                continue
            seen_origins.add(oid)

            for did in comp_origin_to_dests.get(oid, set()):
                if did not in seen_dests:
                    seen_dests.add(did)
                    for next_oid in comp_dest_to_origins.get(did, set()):
                        if next_oid not in seen_origins:
                            stack_origins.append(next_oid)

        unvisited_origins -= seen_origins
        components.append((seen_origins, seen_dests))

    D_target_scaled = pd.Series(0.0, index=D_target.index, dtype='float64')
    dropped_components = 0

    for comp_origins, comp_dests in components:
        O_sum = float(O_target.reindex(list(comp_origins)).fillna(0).sum())
        D_sum = float(D_target.reindex(list(comp_dests)).fillna(0).sum())

        if O_sum <= 0 or D_sum <= 0:
            dropped_components += 1
            continue

        local_scale = O_sum / D_sum
        D_target_scaled.loc[list(comp_dests)] = D_target.reindex(list(comp_dests)).fillna(0) * local_scale

    if dropped_components > 0:
        print(f"  ! UWAGA: Pominięto komponenty bez dodatnich marginesów: {dropped_components}")

    valid_dests = set(D_target_scaled[D_target_scaled > 0].index)
    df = df[df['dest_id'].isin(valid_dests)].copy()

    if df.empty:
        raise ValueError("Brak relacji OD po bilansowaniu komponentowym.")

    O_target = O_target.reindex(df['origin_id'].unique()).fillna(0)
    D_target_scaled = D_target_scaled.reindex(df['dest_id'].unique()).fillna(0)

    # 4. Diagnostyka wykonalności lokalnej i ewentualna augmentacja dummy
    #    Utrzymujemy constrained-constrained, domykając wsparcie w sposób penalizowany kosztowo.
    edges = df[['origin_id', 'dest_id']].drop_duplicates()
    cap_O_by_dest = (
        edges.merge(O_target.rename('O_tmp'), left_on='origin_id', right_index=True, how='left')
        .groupby('dest_id')['O_tmp']
        .sum()
        .reindex(D_target_scaled.index)
        .fillna(0)
    )
    cap_D_by_origin = (
        edges.merge(D_target_scaled.rename('D_tmp'), left_on='dest_id', right_index=True, how='left')
        .groupby('origin_id')['D_tmp']
        .sum()
        .reindex(O_target.index)
        .fillna(0)
    )

    dest_def = (D_target_scaled - cap_O_by_dest).clip(lower=0)
    origin_def = (O_target - cap_D_by_origin).clip(lower=0)

    need_dummy = bool((dest_def > 1e-9).any() or (origin_def > 1e-9).any())

    if need_dummy:
        dummy_origin = '__dummy_origin__'
        dummy_dest = '__dummy_dest__'

        sum_dest_def = float(dest_def.sum())
        sum_origin_def = float(origin_def.sum())
        dummy_mass = max(sum_dest_def, sum_origin_def, 1e-6)

        O_target = O_target.copy()
        D_target_scaled = D_target_scaled.copy()
        O_target.loc[dummy_origin] = dummy_mass
        D_target_scaled.loc[dummy_dest] = dummy_mass

        c95 = float(df['cost'].quantile(0.95)) if len(df) else 60.0
        technical_cost = max(c95 + 30.0, 60.0)

        # Łączymy dummy-origin ze wszystkimi dest i wszystkie origin z dummy-dest,
        # plus krawędź dummy->dummy jako bufor masy technicznej.
        extra_rows = []
        for did in D_target_scaled.index:
            if did == dummy_dest:
                continue
            extra_rows.append((dummy_origin, did))
        for oid in O_target.index:
            if oid == dummy_origin:
                continue
            extra_rows.append((oid, dummy_dest))
        extra_rows.append((dummy_origin, dummy_dest))

        extra_df = pd.DataFrame(extra_rows, columns=['origin_id', 'dest_id']).drop_duplicates()
        extra_df['cost'] = technical_cost

        if isinstance(beta, str) and beta in df.columns and beta in cost_matrix_df.columns:
            beta_map = cost_matrix_df.groupby('origin_id')[beta].first()
            extra_df[beta] = extra_df['origin_id'].map(beta_map).fillna(0.08)
            extra_df['f_cij'] = np.exp(-extra_df[beta] * extra_df['cost'])
        else:
            extra_df['f_cij'] = np.exp(-float(beta) * extra_df['cost'])

        for col in df.columns:
            if col not in extra_df.columns:
                extra_df[col] = np.nan

        extra_df = extra_df[df.columns]
        df = pd.concat([df, extra_df], ignore_index=True)
        print(
            f"  ! INFO: Aktywowano dummy-augmentacje OD "
            f"(def_dest={sum_dest_def:.2f}, def_orig={sum_origin_def:.2f}, mass={dummy_mass:.2f}, edges={len(extra_df)})."
        )

    df['O_i'] = df['origin_id'].map(O_target).fillna(0)
    df['D_j'] = df['dest_id'].map(D_target_scaled).fillna(0)

    A_i = pd.Series(1.0, index=O_target.index, dtype='float64')
    B_j = pd.Series(1.0, index=D_target_scaled.index, dtype='float64')

    # Stabilizacja numeryczna
    damping = 1.0
    eps = 1e-12
    clip_min, clip_max = 1e-12, 1e12
    last_err = np.inf
    marginal_err_max = np.inf
    marginal_err_p95 = np.inf
    marginal_err_wmean = np.inf
    converged = False
    n_iter = 0
    solver_mode = 'doubly_constrained'

    for iteration in range(max_iter):
        n_iter = iteration + 1

        denom_A = (df['dest_id'].map(B_j) * df['D_j'] * df['f_cij']).groupby(df['origin_id']).sum()
        new_A_i = pd.Series(0.0, index=A_i.index, dtype='float64')
        valid_A = denom_A > eps
        new_A_i.loc[valid_A] = 1.0 / denom_A.loc[valid_A]

        denom_B = (df['origin_id'].map(new_A_i) * df['O_i'] * df['f_cij']).groupby(df['dest_id']).sum()
        new_B_j = pd.Series(0.0, index=B_j.index, dtype='float64')
        valid_B = denom_B > eps
        new_B_j.loc[valid_B] = 1.0 / denom_B.loc[valid_B]

        # Skalowanie sprzężone (A/k, B*k) zachowuje A_i*B_j
        pos_A = new_A_i[new_A_i > 0]
        if len(pos_A) > 0:
            k = float(pos_A.mean())
            if np.isfinite(k) and k > 0:
                new_A_i = new_A_i / k
                new_B_j = new_B_j * k

        # Adaptacyjna redukcja kroku przy sygnale oscylacji
        if np.isfinite(last_err) and iteration > 0:
            probe_err = max(float((new_A_i - A_i).abs().max()), float((new_B_j - B_j).abs().max()))
            if probe_err > last_err * 1.5:
                damping = max(0.6, damping * 0.85)
            else:
                damping = min(1.0, damping * 1.03)

        new_A_i = (damping * new_A_i + (1.0 - damping) * A_i).clip(lower=clip_min, upper=clip_max)
        new_B_j = (damping * new_B_j + (1.0 - damping) * B_j).clip(lower=clip_min, upper=clip_max)

        diff_A = float((new_A_i - A_i).abs().max())
        diff_B = float((new_B_j - B_j).abs().max())
        last_err = max(diff_A, diff_B)

        if not np.isfinite(last_err):
            print(f"  ! UWAGA: IPF przerwano (niestabilność numeryczna) po {iteration+1} iteracjach")
            break

        A_i = new_A_i
        B_j = new_B_j

        # Kryterium metodologiczne: zgodność marginaliów O_i / D_j
        if (iteration + 1) % 20 == 0 or iteration == 0:
            T_tmp = df['origin_id'].map(A_i) * df['O_i'] * df['dest_id'].map(B_j) * df['D_j'] * df['f_cij']
            O_hat = T_tmp.groupby(df['origin_id']).sum().reindex(O_target.index).fillna(0)
            D_hat = T_tmp.groupby(df['dest_id']).sum().reindex(D_target_scaled.index).fillna(0)

            rel_O = ((O_hat - O_target).abs() / O_target.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna()
            rel_D = ((D_hat - D_target_scaled).abs() / D_target_scaled.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna()

            if len(rel_O) + len(rel_D) > 0:
                all_rel = pd.concat([rel_O, rel_D], axis=0)
                marginal_err_max = float(all_rel.max())
                marginal_err_p95 = float(all_rel.quantile(0.95))

                w_O = O_target.reindex(rel_O.index).fillna(0).clip(lower=0)
                w_D = D_target_scaled.reindex(rel_D.index).fillna(0).clip(lower=0)
                num = float((rel_O * w_O).sum() + (rel_D * w_D).sum())
                den = float(w_O.sum() + w_D.sum())
                marginal_err_wmean = num / den if den > 0 else float(all_rel.mean())

                if (marginal_err_p95 < 2e-2 and marginal_err_wmean < 5e-3) or (marginal_err_max < max(1e-4, tol * 100)):
                    print(
                        f"  ✓ IPF zbiegł po {iteration+1} iteracjach "
                        f"(p95={marginal_err_p95:.6e}, wmean={marginal_err_wmean:.6e}, max={marginal_err_max:.6e})"
                    )
                    converged = True
                    break

        if last_err < tol:
            print(f"  ✓ IPF zbiegł po {iteration+1} iteracjach (Błąd czynników = {last_err:.6e})")
            converged = True
            break

    if not converged:
        print(
            f"  ! UWAGA: Osiągnięto max_iter ({max_iter}) "
            f"przy błędzie marginaliów p95={marginal_err_p95:.6e}, "
            f"wmean={marginal_err_wmean:.6e}, max={marginal_err_max:.6e}"
        )
        solver_mode = 'doubly_constrained_max_iter'
        print("  ! INFO: Zachowano model dwustronnie ograniczony mimo braku pełnej zbieżności.")

    df['A_i'] = df['origin_id'].map(A_i).fillna(0)
    df['B_j'] = df['dest_id'].map(B_j).fillna(0)
    df['T_ij'] = df['A_i'] * df['O_i'] * df['B_j'] * df['D_j'] * df['f_cij']

    df['solver_mode'] = solver_mode
    df['ipf_converged'] = bool(converged)
    df['ipf_iterations'] = int(n_iter)
    df['ipf_factor_err'] = float(last_err) if np.isfinite(last_err) else np.nan
    df['ipf_marginal_err_p95'] = float(marginal_err_p95) if np.isfinite(marginal_err_p95) else np.nan
    df['ipf_marginal_err_wmean'] = float(marginal_err_wmean) if np.isfinite(marginal_err_wmean) else np.nan
    df['ipf_marginal_err_max'] = float(marginal_err_max) if np.isfinite(marginal_err_max) else np.nan

    # Przywracamy oryginalną kolumnę jobs dla zgodności z dalszym pipeline
    df[D_col] = df['D_j']
    df.drop(columns=['O_i', 'D_j'], inplace=True, errors='ignore')
    return df

### Patch naprawczy IPF (2026-05-01)

Wprowadzono poprawkę stabilności i wykonalności solvera dla pełnych wariantów `dublin` oraz `warsaw`, przy zachowaniu struktury modelu **dwustronnie ograniczonego**.

Zakres zmian:
- dodano etap diagnostyki wykonalności wsparcia OD (maski zer strukturalnych),
- dodano selektywne relacje techniczne OD o wysokim koszcie (tylko tam, gdzie marginesy nie były osiągalne na istniejącym wsparciu),
- ustawiono domyślnie pełny krok iteracyjny (`damping = 1.0`) oraz adaptacyjną redukcję kroku tylko przy oznakach oscylacji,
- pozostawiono kryterium zbieżności oparte na błędach marginaliów (`p95`, `wmean`, `max`).

Interpretacja metodologiczna:
- relacje techniczne mają charakter numerycznego „bezpiecznika” wykonalności i są penalizowane wysokim kosztem,
- nie zmienia to formalnej klasy modelu (nadal constrained-constrained),
- rozwiązanie minimalizuje ryzyko fałszywej rozbieżności wynikającej z niekompletnego wsparcia OD, a nie z samej teorii modelu Wilsona.

In [4]:
def run_city_gravity_pipeline(city, apply_mock_warsaw=False):
    """
    Przeprowadza ciąg analityczny grawitacji na dane miasto w wariancie trójmodelowym:
    - Model A: Sztywny Niski (beta = 0.03)
    - Model B: Sztywny Wysoki (beta = 0.15)
    - Model C: Dynamiczne beta zależne od wskaźnika zamożności.
    """
    print(f"\n{'='*50}\nUruchamianie silnika Grawitacyjnego w podejściu 3-wariantowym: {city.upper()}\n{'='*50}")

    out_dir = OUTPUTS_E4 / city
    (out_dir / "accessibility").mkdir(parents=True, exist_ok=True)
    (out_dir / "calibration").mkdir(parents=True, exist_ok=True)

    wtc_path = INPUTS_E3 / city / "wtc" / "wtc_matrix.parquet"
    units_path = INPUTS_E3 / city / "od" / "od_units.csv"
    origins_geo_path = INPUTS_E3 / city / "od" / "origins.geojson"

    if not wtc_path.exists() or not origins_geo_path.exists():
        print(f"  ✗ Brak plików podstawowych Etapu III dla {city}. Omijam.")
        return

    print(f"  Ładowanie macierzy podróży WTC z Parquet... ({wtc_path.name})")
    df_wtc = pd.read_parquet(wtc_path)
    df_wtc['origin_id'] = df_wtc['origin_id'].astype('string')
    df_wtc['dest_id'] = df_wtc['dest_id'].astype('string')
    wtc_dest_ids = set(df_wtc['dest_id'].dropna().unique())

    if 'cost_min_strict' in df_wtc.columns:
        cost_col = 'cost_min_strict'
    else:
        cost_col = 'cost_min'

    df_wtc = df_wtc.dropna(subset=[cost_col]).copy()
    df_wtc.rename(columns={cost_col: 'cost'}, inplace=True)

    print(f"  Załadowano połączeń o sensownym WTC: {len(df_wtc)}")

    df_od = pd.read_csv(units_path) if units_path.exists() else pd.DataFrame()
    origins_geo = gpd.read_file(origins_geo_path)

    # Paris i paris_core używają kolumny income_median.
    wealth_col = 'income_median' if city.startswith('paris') else 'wealth_index'
    if wealth_col not in origins_geo.columns:
        print(f"  ! UWAGA: Brak kolumny wskaźnika zamożności '{wealth_col}' w origins.geojson. Dynamiczne beta będzie ustawione na wartość domyślną.")
        origins_geo['beta_dynamic'] = 0.08
    else:
        valid_idx = origins_geo[wealth_col].notna()
        if valid_idx.sum() > 0:
            try:
                quantiles = pd.qcut(origins_geo.loc[valid_idx, wealth_col], 5, labels=False, duplicates='drop')
                max_q = quantiles.max() if quantiles.max() > 0 else 1
                beta_mapped = 0.03 + (0.15 - 0.03) * (quantiles / max_q)
                origins_geo.loc[valid_idx, 'beta_dynamic'] = beta_mapped
            except ValueError:
                origins_geo.loc[valid_idx, 'beta_dynamic'] = 0.08

        origins_geo['beta_dynamic'] = origins_geo['beta_dynamic'].fillna(0.08)

    df_origins_geo_sub = origins_geo[['origin_id', 'beta_dynamic']]

    if 'D_jobs' not in df_od.columns:
        df_od['D_jobs'] = np.nan
    if 'O_pop' not in df_od.columns:
        df_od['O_pop'] = np.nan

    origins = df_od[['unit_id', 'O_pop']].rename(columns={'unit_id': 'origin_id'})
    destinations = df_od[['unit_id', 'D_jobs']].rename(columns={'unit_id': 'dest_id'})

    if city == 'warsaw' and apply_mock_warsaw:
        print("  [METODOLOGIA] Warsaw: użycie zastępczych danych o pracy z przygotowanego skryptu.")
        dummy_grid_path = OUTPUTS / "etap1" / "warsaw" / "spatial" / "grid_1km_metric_dummy_jobs.geojson"
        if dummy_grid_path.exists():
            gdf_dummy = gpd.read_file(dummy_grid_path)
            dest_geo_path = INPUTS_E3 / city / "od" / "destinations.geojson"
            if dest_geo_path.exists():
                dest_geo = gpd.read_file(dest_geo_path)
                joined = gpd.sjoin(dest_geo, gdf_dummy[['employment', 'geometry']], how='left', predicate='within')
                joined_agg = joined.groupby('dest_id')['employment'].sum().reset_index()
                destinations = destinations.merge(joined_agg, on='dest_id', how='left')
                destinations['D_jobs'] = destinations['employment']
                destinations.drop(columns=['employment'], inplace=True)
                print("  Mapowanie zapasowego zatrudnienia (employment) udane.")
            else:
                print("  ! Brak destinations.geojson do mapowania !")
        else:
            print("  ! Brak warstwy dummy employment dla Warsaw !")

    if city == 'warsaw' and destinations['D_jobs'].isna().all():
        raise ValueError(
            "Warsaw D_jobs are fully missing. Provide validated employment inputs from Etap I; "
            "random fallback is disabled for methodological correctness."
        )

    merged = df_wtc.merge(origins, on='origin_id', how='inner')
    merged = merged.merge(destinations, on='dest_id', how='inner')
    merged = merged.merge(df_origins_geo_sub, on='origin_id', how='left')

    merged['O_pop'] = merged['O_pop'].fillna(0)
    merged['D_jobs'] = merged['D_jobs'].fillna(0)

    merged = merged[(merged['O_pop'] > 0) & (merged['D_jobs'] > 0) & (merged['cost'] > 0)].copy()

    if len(merged) == 0:
        print("  ✗ Pusta macierz złączeniowa! Brak pasujących O-pop względem D-jobs.")
        return

    models = {
        'A': {'name': 'Model A (beta = 0.03)', 'beta': 0.03},
        'B': {'name': 'Model B (beta = 0.15)', 'beta': 0.15},
        'C': {'name': 'Model C (dynamiczne beta)', 'beta': 'beta_dynamic'},
    }

    access_gdf = origins_geo.copy()

    for m_key, m_info in models.items():
        print(f"  Faza modelowania grawitacyjnego {m_info['name']}...")
        results = doubly_constrained_gravity(merged, 'O_pop', 'D_jobs', beta=m_info['beta'])

        access_df = (
            results.groupby('origin_id')
            .apply(lambda x: (x['D_jobs'] * x['f_cij']).sum(), include_groups=False)
            .reset_index(name=f'accessibility_index_{m_key}')
        )

        access_gdf = access_gdf.merge(access_df, on='origin_id', how='left')

        flows_file = out_dir / "accessibility" / f"flows_{m_key}.parquet"
        results.to_parquet(flows_file, index=False)

    access_gdf['RDI_C_vs_A'] = (
        (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_A'])
        / access_gdf['accessibility_index_A'].replace(0, np.nan)
    )
    access_gdf['RDI_C_vs_B'] = (
        (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_B'])
        / access_gdf['accessibility_index_B'].replace(0, np.nan)
    )

    access_geo_file = out_dir / "accessibility" / "accessibility_grid_3models.geojson"
    access_gdf.to_file(access_geo_file, driver="GeoJSON")

    calib = {
        'city': city,
        'models': models,
        'n_origins': int(access_gdf.shape[0]),
        'n_flows': int(merged.shape[0]),
        'matrix_used': cost_col,
        'wealth_metric_used': wealth_col,
        'calculation_time': str(datetime.utcnow()),
    }
    with open(out_dir / "calibration" / "beta_params_3models.json", 'w', encoding='utf-8') as f:
        json.dump(calib, f, indent=2)

    print(f"  ✓ Utworzono 3 warianty modeli dla {city.upper()}. Wyeksportowano warstwy Parquet i GeoJSON.")

In [5]:
def run_city_gravity_pipeline(city, apply_mock_warsaw=False):
    """Stabilna wersja pipeline Etapu IV z priorytetem D_jobs/jobs i dosilaniem z Etapu I."""
    print(f"\n{'='*50}\n=== ETAP IV: MODELE GRAWITYCYJNE DLA {city.upper()} ===")
    city_key = city.replace('_core', '').replace('_extended', '')
    _variant = 'extended' if city.endswith('_extended') else 'core'
    _inputs_e3 = OUTPUTS / "etap3" / _variant
    _outputs_e4 = OUTPUTS / "etap4" / _variant

    out_dir = _outputs_e4 / city
    (out_dir / "accessibility").mkdir(parents=True, exist_ok=True)
    (out_dir / "calibration").mkdir(parents=True, exist_ok=True)

    wtc_path = _inputs_e3 / city / "wtc" / "wtc_matrix.parquet"
    units_path = _inputs_e3 / city / "od" / "od_units.csv"
    origins_geo_path = _inputs_e3 / city / "od" / "origins.geojson"
    destinations_geo_path = _inputs_e3 / city / "od" / "destinations.geojson"
    destinations_lookup_path = _inputs_e3 / city / "od" / "destinations_lookup.csv"
    origins_lookup_path = _inputs_e3 / city / "od" / "origins_lookup.csv"

    if not wtc_path.exists() or not origins_geo_path.exists():
        print(f"  ✗ Brak plikow podstawowych Etapu III dla {city}. Omijam.")
        return

    print(f"  Ladowanie macierzy podrozy WTC z Parquet... ({wtc_path.name})")
    df_wtc = pd.read_parquet(wtc_path)

    cost_col = 'cost_min_strict' if 'cost_min_strict' in df_wtc.columns else 'cost_min'
    df_wtc = df_wtc.dropna(subset=[cost_col]).copy()
    df_wtc.rename(columns={cost_col: 'cost'}, inplace=True)
    df_wtc['origin_id'] = df_wtc['origin_id'].astype('string')
    df_wtc['dest_id'] = df_wtc['dest_id'].astype('string')

    wtc_origin_ids = set(df_wtc['origin_id'].dropna().unique())
    wtc_dest_ids = set(df_wtc['dest_id'].dropna().unique())
    print(f"  Zaladowano polaczen o sensownym WTC: {len(df_wtc)}")

    df_od = pd.read_csv(units_path) if units_path.exists() else pd.DataFrame()
    origins_geo = gpd.read_file(origins_geo_path)
    destinations_geo = gpd.read_file(destinations_geo_path) if destinations_geo_path.exists() else None

    city_base = city_key

    def _candidate_stage1_paths(role: str) -> list:
        spatial_dir = OUTPUTS / "etap1" / _variant / city_base / "spatial"
        if not spatial_dir.exists():
            spatial_dir = OUTPUTS / "etap1" / city_base / "spatial"
        if city_base == 'dublin':
            if role == 'destinations':
                return [spatial_dir / "workplace_density_1km.geojson", spatial_dir / "grid_1km_metric.geojson"]
            return [spatial_dir / "grid_1km_metric.geojson", spatial_dir / "grid_1km_metric.geojson.bak_pre_dublin_wealth"]
        if city_base == 'paris':
            return [
                spatial_dir / "workplace_density_1km.geojson" if role == 'destinations' else spatial_dir / "pop_grid_1km_metric.geojson",
                spatial_dir / "pop_grid_1km_metric.geojson",
                spatial_dir / "pop_grid_1km_metric.geojson.bak",
            ]
        if city_base == 'warsaw':
            return [spatial_dir / "grid_1km_metric.geojson", spatial_dir / "grid_1km_metric_dummy_jobs.geojson", spatial_dir / "grid_1km_metric.geojson.bak"]
        return []

    def _merge_stage1_attributes(target_gdf: gpd.GeoDataFrame, role: str, needed_cols: list):
        missing_cols = [c for c in needed_cols if c not in target_gdf.columns or target_gdf[c].notna().sum() == 0]
        if not missing_cols:
            return target_gdf, None

        for stage1_path in _candidate_stage1_paths(role):
            if not stage1_path.exists():
                continue
            stage1_gdf = gpd.read_file(stage1_path)

            if 'wealth_index' in missing_cols and 'wealth_index' not in stage1_gdf.columns and 'hp_index_abs' in stage1_gdf.columns:
                stage1_gdf = stage1_gdf.copy()
                stage1_gdf['wealth_index'] = stage1_gdf['hp_index_abs']

            available_cols = [c for c in missing_cols if c in stage1_gdf.columns]
            if not available_cols:
                continue

            join_keys = None
            left = target_gdf.copy()
            right = stage1_gdf.copy()

            if city_base == 'dublin':
                if 'ITM1KM' in left.columns and 'ITM1KM' in right.columns:
                    join_keys = ['ITM1KM']
                elif 'OBJECTID' in left.columns and 'OBJECTID' in right.columns:
                    join_keys = ['OBJECTID']
            elif city_base == 'paris':
                if 'cell_id' in left.columns and 'cell_id' in right.columns:
                    join_keys = ['cell_id']
                elif {'X', 'Y'}.issubset(left.columns) and {'X', 'Y'}.issubset(right.columns):
                    for frame in (left, right):
                        frame['_join_x'] = pd.to_numeric(frame['X'], errors='coerce').round(6)
                        frame['_join_y'] = pd.to_numeric(frame['Y'], errors='coerce').round(6)
                    join_keys = ['_join_x', '_join_y']
            elif city_base == 'warsaw':
                if 'code' in left.columns and 'code' in right.columns:
                    join_keys = ['code']
                elif 'pl_code' in left.columns and 'pl_code' in right.columns:
                    join_keys = ['pl_code']

            if not join_keys:
                if role == 'destinations' and city_base == 'paris' and left.geometry is not None and right.geometry is not None:
                    left_crs = left.crs
                    right_crs = right.crs
                    if left_crs and right_crs and left_crs != right_crs:
                        right = right.to_crs(left_crs)
                    if 'dest_id' not in left.columns:
                        continue
                    left_pts = left[['dest_id', 'geometry']].copy()
                    if left_pts.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).any():
                        left_pts = left_pts.copy()
                        left_pts['geometry'] = left_pts.geometry.centroid
                    right_poly = right[available_cols + ['geometry']].copy()
                    joined = gpd.sjoin(left_pts, right_poly, how='left', predicate='within')
                    joined = joined.drop_duplicates(subset=['dest_id'])
                    used_cols = []
                    for col in available_cols:
                        if col in joined.columns:
                            idx_map = joined.set_index('dest_id')[col]
                            enriched_vals = target_gdf['dest_id'].map(idx_map)
                            if col not in target_gdf.columns:
                                target_gdf[col] = enriched_vals
                            else:
                                target_gdf[col] = target_gdf[col].where(target_gdf[col].notna(), enriched_vals)
                            if target_gdf[col].notna().sum() > 0:
                                used_cols.append(col)
                    if used_cols:
                        return target_gdf, f"{_rel(stage1_path)}:{','.join(used_cols)} (spatial join)"
                continue

            merge_cols = join_keys + available_cols
            merged_attr = left.merge(right[merge_cols].drop_duplicates(subset=join_keys), on=join_keys, how='left', suffixes=('', '_stage1'))
            used_cols = []
            for col in available_cols:
                col_stage1 = f'{col}_stage1' if f'{col}_stage1' in merged_attr.columns else col
                if col not in target_gdf.columns:
                    target_gdf[col] = merged_attr[col_stage1]
                else:
                    target_gdf[col] = target_gdf[col].where(target_gdf[col].notna(), merged_attr[col_stage1])
                if target_gdf[col].notna().sum() > 0:
                    used_cols.append(col)

            if used_cols:
                return target_gdf, f"{_rel(stage1_path)}:{','.join(used_cols)}"

        return target_gdf, None

    wealth_col = 'income_median' if city.startswith('paris') else 'wealth_index'
    origins_geo, wealth_source = _merge_stage1_attributes(origins_geo, 'origins', [wealth_col])
    if destinations_geo is not None:
        destinations_geo, _ = _merge_stage1_attributes(destinations_geo, 'destinations', ['D_jobs', 'jobs', 'employment'])

    if wealth_col not in origins_geo.columns or origins_geo[wealth_col].notna().sum() == 0:
        print(f"  ! UWAGA: Brak kolumny wskaznika zamoznosci '{wealth_col}' w Etapie III i artefaktach Etapu I. Dynamiczne beta bedzie ustawione na wartosc domyslna.")
        origins_geo['beta_dynamic'] = 0.08
    else:
        if wealth_source:
            print(f"  Uzupelniono wskaznik zamoznosci '{wealth_col}' z {wealth_source}.")
        valid_idx = origins_geo[wealth_col].notna()
        if valid_idx.sum() > 0:
            try:
                q = pd.qcut(origins_geo.loc[valid_idx, wealth_col], 5, labels=False, duplicates='drop')
                max_q = q.max() if q.max() > 0 else 1
                origins_geo.loc[valid_idx, 'beta_dynamic'] = 0.03 + (0.15 - 0.03) * (q / max_q)
            except ValueError:
                origins_geo.loc[valid_idx, 'beta_dynamic'] = 0.08
        origins_geo['beta_dynamic'] = origins_geo['beta_dynamic'].fillna(0.08)

    def _score_candidate(df_cand: pd.DataFrame, id_col: str, val_col: str, wtc_ids: set, src_name: str):
        d = df_cand.copy()
        d[id_col] = d[id_col].astype('string')
        d[val_col] = pd.to_numeric(d[val_col], errors='coerce').fillna(0)
        d = d.dropna(subset=[id_col]).groupby(id_col, as_index=False)[val_col].first()

        ids = set(d[id_col].dropna().unique())
        overlap_ids = ids.intersection(wtc_ids)
        d_pos = d[d[val_col] > 0]
        pos_ids = set(d_pos[id_col].dropna().unique())
        overlap_pos = pos_ids.intersection(wtc_ids)

        semantic_priority = 1 if any(k in src_name for k in ['D_jobs', 'jobs', 'employment']) else 0
        score = (int(semantic_priority), int(len(overlap_pos)), int(len(overlap_ids)), int(len(pos_ids)), int(len(ids)))
        return d, score

    def _load_origins():
        candidates = []
        c_geo = _pick_first_column(origins_geo, ['O_pop', 'pop'])
        if c_geo and 'origin_id' in origins_geo.columns:
            candidates.append((origins_geo[['origin_id', c_geo]].rename(columns={c_geo: 'O_pop'}), f'origins_geo:{c_geo}'))

        if origins_lookup_path.exists():
            d = pd.read_csv(origins_lookup_path)
            c = _pick_first_column(d, ['O_pop', 'pop'])
            if c and 'origin_id' in d.columns:
                candidates.append((d[['origin_id', c]].rename(columns={c: 'O_pop'}), f'origins_lookup:{c}'))

        if {'unit_id', 'O_pop', 'role'}.issubset(df_od.columns):
            d = df_od[df_od['role'].astype(str).str.lower() == 'origin'][['unit_id', 'O_pop']]
            candidates.append((d.rename(columns={'unit_id': 'origin_id'}), 'od_units:O_pop'))

        best_df, best_source, best_score = None, None, None
        for df_cand, source in candidates:
            scored_df, score = _score_candidate(df_cand, 'origin_id', 'O_pop', wtc_origin_ids, source)
            if best_score is None or score > best_score:
                best_df, best_source, best_score = scored_df, source, score
        return best_df, best_source

    def _load_destinations():
        candidates = []

        if destinations_geo is not None:
            c = _pick_first_column(destinations_geo, ['D_jobs', 'jobs', 'employment'])
            if c and 'dest_id' in destinations_geo.columns:
                candidates.append((destinations_geo[['dest_id', c]].rename(columns={c: 'D_jobs'}), f'destinations_geo:{c}'))

        if destinations_lookup_path.exists():
            d = pd.read_csv(destinations_lookup_path)
            c_jobs = _pick_first_column(d, ['D_jobs', 'jobs', 'employment'])
            if c_jobs and 'dest_id' in d.columns:
                candidates.append((d[['dest_id', c_jobs]].rename(columns={c_jobs: 'D_jobs'}), f'destinations_lookup:{c_jobs}'))
            c_pop = _pick_first_column(d, ['pop'])
            if c_pop and 'dest_id' in d.columns:
                candidates.append((d[['dest_id', c_pop]].rename(columns={c_pop: 'D_jobs'}), f'destinations_lookup:{c_pop}'))

        if {'unit_id', 'role'}.issubset(df_od.columns):
            c_jobs = _pick_first_column(df_od, ['D_jobs', 'jobs', 'employment'])
            if c_jobs:
                d_jobs = df_od[df_od['role'].astype(str).str.lower().isin(['dest', 'destination'])][['unit_id', c_jobs]]
                candidates.append((d_jobs.rename(columns={'unit_id': 'dest_id', c_jobs: 'D_jobs'}), f'od_units:{c_jobs}'))
            c_pop = _pick_first_column(df_od, ['pop'])
            if c_pop:
                d_pop = df_od[df_od['role'].astype(str).str.lower().isin(['dest', 'destination'])][['unit_id', c_pop]]
                candidates.append((d_pop.rename(columns={'unit_id': 'dest_id', c_pop: 'D_jobs'}), f'od_units:{c_pop}'))

        best_df, best_source, best_score = None, None, None
        for df_cand, source in candidates:
            scored_df, score = _score_candidate(df_cand, 'dest_id', 'D_jobs', wtc_dest_ids, source)
            if best_score is None or score > best_score:
                best_df, best_source, best_score = scored_df, source, score

        if best_source and ':pop' in best_source:
            print("  ! UWAGA: Brak D_jobs/jobs/employment; uzywam pop jako ostateczny proxy atrakcyjnosci celu.")

        return best_df, best_source

    origins, origins_source = _load_origins()
    destinations, destinations_source = _load_destinations()

    if origins is None or destinations is None:
        print("  ✗ Nie udalo sie zbudowac marginesow O_pop/D_jobs z plikow Etapu III/I. Omijam.")
        return

    origins = origins[origins['origin_id'].astype('string').isin(wtc_origin_ids)].copy()
    destinations = destinations[destinations['dest_id'].astype('string').isin(wtc_dest_ids)].copy()

    origins['origin_id'] = origins['origin_id'].astype('string')
    destinations['dest_id'] = destinations['dest_id'].astype('string')
    origins = origins.dropna(subset=['origin_id']).groupby('origin_id', as_index=False)['O_pop'].first()
    destinations = destinations.dropna(subset=['dest_id']).groupby('dest_id', as_index=False)['D_jobs'].first()

    merged = df_wtc.merge(origins, on='origin_id', how='inner')
    merged = merged.merge(destinations, on='dest_id', how='inner')
    merged['O_pop'] = pd.to_numeric(merged['O_pop'], errors='coerce').fillna(0)
    merged['D_jobs'] = pd.to_numeric(merged['D_jobs'], errors='coerce').fillna(0)
    merged = merged[(merged['O_pop'] > 0) & (merged['D_jobs'] > 0) & (merged['cost'] > 0)].copy()

    if len(merged) == 0:
        print("  ✗ Pusta macierz po polaczeniu z marginesami. Omijam.")
        return

    df_origins_geo_sub = origins_geo[['origin_id', 'beta_dynamic']].copy()
    df_origins_geo_sub['origin_id'] = df_origins_geo_sub['origin_id'].astype('string')

    access_gdf = origins_geo[['origin_id', 'geometry']].copy()
    access_gdf['origin_id'] = access_gdf['origin_id'].astype('string')

    models = {
        'A': {'name': 'Model A (beta = 0.03)', 'beta': 0.03},
        'B': {'name': 'Model B (beta = 0.15)', 'beta': 0.15},
        'C': {'name': 'Model C (dynamiczne beta)', 'beta': 'beta_dynamic'},
    }
    model_diagnostics = {}

    for m_key, m_info in models.items():
        print(f"  Faza modelowania grawitacyjnego {m_info['name']}...")

        if m_info['beta'] == 'beta_dynamic':
            merged_m = merged.merge(df_origins_geo_sub, on='origin_id', how='left')
            merged_m['beta_dynamic'] = merged_m['beta_dynamic'].fillna(0.08)
        else:
            merged_m = merged.copy()
            merged_m['beta_dynamic'] = float(m_info['beta'])

        merged_m['f_cij'] = np.exp(-merged_m['beta_dynamic'] * merged_m['cost'])
        results = doubly_constrained_gravity(merged_m, 'O_pop', 'D_jobs', beta=m_info['beta'])

        solver_mode = str(results['solver_mode'].iloc[0]) if 'solver_mode' in results.columns and len(results) else 'unknown'
        ipf_converged = bool(results['ipf_converged'].iloc[0]) if 'ipf_converged' in results.columns and len(results) else False
        ipf_iterations = int(results['ipf_iterations'].iloc[0]) if 'ipf_iterations' in results.columns and len(results) else None
        err_p95 = float(results['ipf_marginal_err_p95'].iloc[0]) if 'ipf_marginal_err_p95' in results.columns and len(results) else np.nan
        err_wmean = float(results['ipf_marginal_err_wmean'].iloc[0]) if 'ipf_marginal_err_wmean' in results.columns and len(results) else np.nan
        err_max = float(results['ipf_marginal_err_max'].iloc[0]) if 'ipf_marginal_err_max' in results.columns and len(results) else np.nan

        model_diagnostics[m_key] = {
            'solver_mode': solver_mode,
            'ipf_converged': ipf_converged,
            'ipf_iterations': ipf_iterations,
            'ipf_marginal_err_p95': err_p95,
            'ipf_marginal_err_wmean': err_wmean,
            'ipf_marginal_err_max': err_max,
        }

        access_gdf[f'solver_mode_{m_key}'] = solver_mode
        access_gdf[f'ipf_converged_{m_key}'] = int(ipf_converged)

        access_df = (
            results.groupby('origin_id')
            .apply(lambda x: (x['D_jobs'] * x['f_cij']).sum(), include_groups=False)
            .reset_index(name=f'accessibility_index_{m_key}')
        )
        access_gdf = access_gdf.merge(access_df, on='origin_id', how='left')

        flows_file = out_dir / "accessibility" / f"flows_{m_key}.parquet"
        results.to_parquet(flows_file, index=False)

    access_gdf['RDI_C_vs_A'] = (
        (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_A'])
        / access_gdf['accessibility_index_A'].replace(0, np.nan)
    )
    access_gdf['RDI_C_vs_B'] = (
        (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_B'])
        / access_gdf['accessibility_index_B'].replace(0, np.nan)
    )

    geojson_file = out_dir / "accessibility" / "accessibility_grid_3models.geojson"
    access_gdf.to_file(geojson_file, driver='GeoJSON')

    beta_params = {
        'city': city,
        'models': {k: {'name': v['name'], 'beta': v['beta']} for k, v in models.items()},
        'n_origins': len(origins),
        'n_flows': len(merged),
        'matrix_used': cost_col,
        'wealth_metric_used': wealth_col,
        'wealth_metric_source': wealth_source,
        'origins_source': origins_source,
        'destinations_source': destinations_source,
        'model_diagnostics': model_diagnostics,
        'calculation_time': str(pd.Timestamp.now()),
    }
    with open(out_dir / "calibration" / "beta_params_3models.json", 'w', encoding='utf-8') as f:
        json.dump(beta_params, f, indent=2, ensure_ascii=False)

    print(f"  ✓ Utworzono 3 warianty modeli dla {city.upper()}. Wyeksportowano warstwy Parquet i GeoJSON.")

In [6]:
# run_city_gravity_pipeline('dublin', apply_mock_warsaw=False)
# run_city_gravity_pipeline('paris', apply_mock_warsaw=False)
# run_city_gravity_pipeline('warsaw', apply_mock_warsaw=False)

In [7]:
# Przeprowadzenie silnika grawitacji na każdą metropolię w nowych standardach 3-wariantowych
for c in CITIES:
    # Używamy False, ponieważ Warszawa ma już prawdziwe dane z Etapu 1 (waw_miejsca_pracy.shp)
    run_city_gravity_pipeline(c, apply_mock_warsaw=False)



=== ETAP IV: MODELE GRAWITYCYJNE DLA PARIS_CORE ===
  Ladowanie macierzy podrozy WTC z Parquet... (wtc_matrix.parquet)
  Zaladowano polaczen o sensownym WTC: 652869
  Faza modelowania grawitacyjnego Model A (beta = 0.03)...
  ✓ IPF zbiegł po 20 iteracjach (p95=8.438120e-06, wmean=8.339733e-06, max=2.658625e-02)
  Faza modelowania grawitacyjnego Model B (beta = 0.15)...
  ✓ IPF zbiegł po 20 iteracjach (p95=1.060306e-03, wmean=1.497265e-04, max=1.150887e-02)
  Faza modelowania grawitacyjnego Model C (dynamiczne beta)...
  ✓ IPF zbiegł po 20 iteracjach (p95=1.227828e-05, wmean=8.537728e-06, max=2.447231e-02)
  ✓ Utworzono 3 warianty modeli dla PARIS_CORE. Wyeksportowano warstwy Parquet i GeoJSON.

=== ETAP IV: MODELE GRAWITYCYJNE DLA DUBLIN_CORE ===
  Ladowanie macierzy podrozy WTC z Parquet... (wtc_matrix.parquet)
  Zaladowano polaczen o sensownym WTC: 13066
  Faza modelowania grawitacyjnego Model A (beta = 0.03)...
  ! INFO: Aktywowano dummy-augmentacje OD (def_dest=139.09, def_orig=0.

In [8]:
city = 'paris_core'
_variant = 'extended' if city.endswith('_extended') else 'core'
_inputs_e3 = OUTPUTS / 'etap3' / _variant
wtc_path = _inputs_e3 / city / 'wtc' / 'wtc_matrix.parquet'
units_path = _inputs_e3 / city / 'od' / 'od_units.csv'
origins_geo_path = _inputs_e3 / city / 'od' / 'origins.geojson'
print('wtc', wtc_path, wtc_path.exists())
print('units', units_path, units_path.exists())
print('orig', origins_geo_path, origins_geo_path.exists())

wtc C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\core\paris_core\wtc\wtc_matrix.parquet True
units C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\core\paris_core\od\od_units.csv False
orig C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\core\paris_core\od\origins.geojson True
